In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("dataanalyst2.csv")
df.head()

,Link,Job Title,Company,Salary,Location,Experience,Skills,Day Posted
0,https://www.naukri.com/job-listings-data-analy...,Data Analyst,AU Small Finance Bank,4-6 Lacs PA,Bengaluru,0-2 Yrs,"Analytics, R Studio, Data Extraction, Advanced...",2022-01
1,https://www.naukri.com/job-listings-client-dat...,Client Data Analyst,JPMorgan Chase Bank,Not disclosed,Bengaluru,0-5 Yrs,"Service delivery, Senior management, Product s...",2022-01
2,https://www.naukri.com/job-listings-data-analy...,"Data Analyst, EasyShip",Amazon,Not disclosed,Bengaluru,0-7 Yrs,"Automation, Data analysis, IT Analyst, Analyti...",2022-01
3,https://www.naukri.com/job-listings-data-analy...,"Data Analyst, FCGT, Local Shops on Amazon",Amazon,Not disclosed,Bengaluru,0-7 Yrs,"Automation, SASBusiness analysis, Operations p...",2022-01
4,https://www.naukri.com/job-listings-data-analy...,"Data Analyst, Alexa Audio",Amazon,Not disclosed,Chennai,0-7 Yrs,"Qualitative research, Usage, Web technologies,...",2022-01


In [3]:
df.shape

(4191, 8)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4191 entries, 0 to 4190
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Link        4191 non-null   object
 1   Job Title   4191 non-null   object
 2   Company     4191 non-null   object
 3   Salary      4191 non-null   object
 4   Location    4191 non-null   object
 5   Experience  4191 non-null   object
 6   Skills      4191 non-null   object
 7   Day Posted  4191 non-null   object
dtypes: object(8)
memory usage: 262.1+ KB


In [5]:
df.describe()

,Link,Job Title,Company,Salary,Location,Experience,Skills,Day Posted
count,4191,4191,4191,4191,4191,4191,4191,4191
unique,78,32,64,15,37,29,75,36
top,https://www.naukri.com/job-listings-data-analy...,Data Analyst,Capgemini,Not disclosed,Bengaluru,1-6 Yrs,"Palantir platform, JAVA, Data analytics tool A...",2024-08
freq,158,2727,417,3558,992,473,158,261


In [6]:
print(df['Salary'].value_counts())
(df['Experience'].value_counts())

Salary
Not disclosed       3558
3-3.5 Lacs PA        158
4-9 Lacs PA          158
4-8 Lacs PA          157
2.25-3 Lacs PA       100
12-20 Lacs PA         50
3.75-6 Lacs PA         2
4-6 Lacs PA            1
4-5 Lacs PA            1
1-3 Lacs PA            1
5-7 Lacs PA            1
3.75-5.5 Lacs PA       1
6-9 Lacs PA            1
15-30 Lacs PA          1
15-25 Lacs PA          1
Name: count, dtype: int64


Experience
1-6 Yrs      473
2-4 Yrs      468
3-5 Yrs      466
0-1 Yrs      418
3-8 Yrs      210
2-5 Yrs      209
5-9 Yrs      208
4-8 Yrs      207
2-7 Yrs      160
5-10 Yrs     159
4-6 Yrs      158
3-7 Yrs      158
1-3 Yrs      158
1-5 Yrs      158
1-4 Yrs      158
0-3 Yrs      101
3-6 Yrs      100
4-9 Yrs       51
6-9 Yrs       50
5-8 Yrs       50
4-5 Yrs       50
0-5 Yrs        8
0-7 Yrs        4
0-2 Yrs        4
0-6 Yrs        1
10-20 Yrs      1
7-9 Yrs        1
7-12 Yrs       1
7-10 Yrs       1
Name: count, dtype: int64

In [7]:
import numpy as np
import re

# Function to extract numeric salary values from string
def extract_salary(s):
    numbers = re.findall(r'\d+', s)  # Extract numbers from salary string
    return list(map(float, numbers)) if numbers else None

# Compute median salary for each experience range
salary_dict = {}

for exp in df['Experience'].unique():
    salaries = [np.mean(extract_salary(s)) for s in df.loc[df['Experience'] == exp, 'Salary'] if extract_salary(s)]
    
    if salaries:  # If salaries exist for this experience range
        salary_dict[exp] = np.median(salaries)

# Compute overall median salary for fallback
overall_median_salary = np.median([np.mean(extract_salary(s)) for s in df['Salary'] if extract_salary(s)])

# Function to find the closest experience range if exact match is missing
def find_closest_experience(exp):
    exp_numbers = [int(num) for num in re.findall(r'\d+', exp)]
    exp_years = (min(exp_numbers), max(exp_numbers)) if exp_numbers else (None, None)

    # Find an alternative experience range if current one is missing in salary_dict
    for key in salary_dict.keys():
        key_numbers = [int(num) for num in re.findall(r'\d+', key)]
        key_years = (min(key_numbers), max(key_numbers)) if key_numbers else (None, None)

        # If ranges overlap, return that experience level
        if exp_years[0] <= key_years[1] and exp_years[1] >= key_years[0]:
            return key

    return None  # No similar experience range found

# Function to replace "Not disclosed" salaries
def impute_salary(row):
    if row['Salary'] == 'Not disclosed':
        experience = row['Experience']
        
        # Use exact experience-based median salary if available
        if experience in salary_dict:
            median_salary = salary_dict[experience]
        else:
            # Find closest experience range
            closest_exp = find_closest_experience(experience)
            if closest_exp:
                median_salary = salary_dict[closest_exp]
            else:
                median_salary = overall_median_salary  # Fallback to overall median

        return f"{int(median_salary)-2}-{int(median_salary)+2} Lacs PA"
    
    return row['Salary']

def impute_salary1(row):
    if row['Salary'] == 'Unpaid':
        experience = row['Experience']
        
        # Use exact experience-based median salary if available
        if experience in salary_dict:
            median_salary = salary_dict[experience]
        else:
            # Find closest experience range
            closest_exp = find_closest_experience(experience)
            if closest_exp:
                median_salary = salary_dict[closest_exp]
            else:
                median_salary = overall_median_salary  # Fallback to overall median

        return f"{int(median_salary)-2}-{int(median_salary)+2} Lacs PA"
    
    return row['Salary']


# Apply the imputation
df['Salary'] = df.apply(impute_salary, axis=1)
df['Salary'] = df.apply(impute_salary1, axis=1)

# Check if "Not disclosed" values are reduced
print(df['Salary'].value_counts())


Salary
2-6 Lacs PA         1582
4-8 Lacs PA          989
26-30 Lacs PA        777
1-5 Lacs PA          159
18-22 Lacs PA        158
4-9 Lacs PA          158
3-3.5 Lacs PA        158
2.25-3 Lacs PA       100
12-20 Lacs PA         50
14-18 Lacs PA         50
3.75-6 Lacs PA         2
15-25 Lacs PA          1
15-30 Lacs PA          1
4-6 Lacs PA            1
3.75-5.5 Lacs PA       1
5-7 Lacs PA            1
1-3 Lacs PA            1
4-5 Lacs PA            1
6-9 Lacs PA            1
Name: count, dtype: int64


In [8]:
df[['Salary','Experience']]

,Salary,Experience
0,4-6 Lacs PA,0-2 Yrs
1,2-6 Lacs PA,0-5 Yrs
2,2-6 Lacs PA,0-7 Yrs
3,2-6 Lacs PA,0-7 Yrs
4,2-6 Lacs PA,0-7 Yrs
...,...,...
4186,2-6 Lacs PA,2-5 Yrs
4187,4-9 Lacs PA,2-7 Yrs
4188,2-6 Lacs PA,1-6 Yrs
4189,2-6 Lacs PA,1-6 Yrs


In [9]:
df.head()

,Link,Job Title,Company,Salary,Location,Experience,Skills,Day Posted
0,https://www.naukri.com/job-listings-data-analy...,Data Analyst,AU Small Finance Bank,4-6 Lacs PA,Bengaluru,0-2 Yrs,"Analytics, R Studio, Data Extraction, Advanced...",2022-01
1,https://www.naukri.com/job-listings-client-dat...,Client Data Analyst,JPMorgan Chase Bank,2-6 Lacs PA,Bengaluru,0-5 Yrs,"Service delivery, Senior management, Product s...",2022-01
2,https://www.naukri.com/job-listings-data-analy...,"Data Analyst, EasyShip",Amazon,2-6 Lacs PA,Bengaluru,0-7 Yrs,"Automation, Data analysis, IT Analyst, Analyti...",2022-01
3,https://www.naukri.com/job-listings-data-analy...,"Data Analyst, FCGT, Local Shops on Amazon",Amazon,2-6 Lacs PA,Bengaluru,0-7 Yrs,"Automation, SASBusiness analysis, Operations p...",2022-01
4,https://www.naukri.com/job-listings-data-analy...,"Data Analyst, Alexa Audio",Amazon,2-6 Lacs PA,Chennai,0-7 Yrs,"Qualitative research, Usage, Web technologies,...",2022-01


In [10]:
df['Job Title'] = 'Data Analyst'

In [11]:
df['Location'].value_counts()

Location
Bengaluru                                                          992
Chennai                                                            632
Gurugram                                                           365
Hyderabad                                                          258
Remote                                                             212
Hybrid - Hyderabad, Bengaluru, Mumbai (All Areas)                  158
Pune, Bengaluru                                                    158
Hybrid - Mumbai                                                    158
Pune(Bavdhan)                                                      158
New Delhi                                                          157
Noida                                                              157
Hosur                                                              157
Kolkata, Mumbai, New Delhi, Hyderabad, Pune, Chennai, Bengaluru    157
Pune                                                                

In [12]:
df['Location'] = df['Location'].replace({'Hybrid - Hyderabad, Pune, Bengaluru':'Hyderabad, Pune, Bengaluru',
                                         'Remote':'Noida','Delhi / NCR, Bengaluru':'Delhi, Bengaluru','Mumbai, Delhi / NCR, Bengaluru':'Mumbai, Delhi, Bengaluru',
                                         'Hybrid - Bengaluru':'Bengaluru','Hybrid - Gurugram':'Delhi','Bengaluru(Electronic City +3)':'Bengaluru',
                                         'Hybrid - Pune, Bengaluru, Mumbai (All Areas)':'Pune, Bengaluru, Mumbai','Pune, Mumbai (All Areas)':'Pune, Mumbai',
                                         'Hybrid - Hyderabad, Chennai, Bengaluru':'Hyderabad, Chennai, Bengaluru','Mumbai (All Areas)':'Mumbai',
                                         'Pune(Bavdhan)':'Pune'
                                         })
df.head()


,Link,Job Title,Company,Salary,Location,Experience,Skills,Day Posted
0,https://www.naukri.com/job-listings-data-analy...,Data Analyst,AU Small Finance Bank,4-6 Lacs PA,Bengaluru,0-2 Yrs,"Analytics, R Studio, Data Extraction, Advanced...",2022-01
1,https://www.naukri.com/job-listings-client-dat...,Data Analyst,JPMorgan Chase Bank,2-6 Lacs PA,Bengaluru,0-5 Yrs,"Service delivery, Senior management, Product s...",2022-01
2,https://www.naukri.com/job-listings-data-analy...,Data Analyst,Amazon,2-6 Lacs PA,Bengaluru,0-7 Yrs,"Automation, Data analysis, IT Analyst, Analyti...",2022-01
3,https://www.naukri.com/job-listings-data-analy...,Data Analyst,Amazon,2-6 Lacs PA,Bengaluru,0-7 Yrs,"Automation, SASBusiness analysis, Operations p...",2022-01
4,https://www.naukri.com/job-listings-data-analy...,Data Analyst,Amazon,2-6 Lacs PA,Chennai,0-7 Yrs,"Qualitative research, Usage, Web technologies,...",2022-01


In [13]:
df['Location'].value_counts()

Location
Bengaluru                                                          992
Chennai                                                            632
Noida                                                              369
Gurugram                                                           365
Hyderabad                                                          258
Pune                                                               210
Hybrid - Hyderabad, Bengaluru, Mumbai (All Areas)                  158
Pune, Bengaluru                                                    158
Hybrid - Mumbai                                                    158
Kolkata, Mumbai, New Delhi, Hyderabad, Pune, Chennai, Bengaluru    157
Hosur                                                              157
New Delhi                                                          157
India                                                               52
Bengaluru(Bannerghatta Road)                                        

In [14]:
df['Day Posted'].value_counts()

Day Posted
2024-08    261
2023-07    250
2022-09    240
2022-03    210
2024-03    204
2023-02    190
2024-07    180
2022-06    150
2024-02    148
2024-11    136
2024-12    136
2022-07    135
2022-02    130
2023-08    130
2023-03    130
2023-10    125
2022-08    120
2022-11    115
2024-09    115
2023-06    111
2024-06    103
2022-10    100
2024-01     99
2023-11     95
2024-10     91
2022-04     80
2023-01     80
2022-01     59
2024-04     51
2024-05     51
2023-04     50
2023-12     46
2022-12     25
2023-05     20
2023-09     15
2022-05     10
Name: count, dtype: int64

In [16]:
df['Skills'] = df['Skills'].astype(str)  # Ensure it's a string
df['Location'] = df['Location'].astype(str)  # Ensure it's a string
df['Skills List'] = df['Skills'].str.split(', ')  # Split skills into a list
df['Location List'] = df['Location'].str.split(', ')  

# Expand skills into multiple rows (optional, if needed for analysis)
df= df.explode('Skills List')
df= df.explode('Location List')

In [17]:
df['Skills List'].value_counts().head(10)
df['Skills List']= df['Skills List'].replace({'Analytical':'Python','Automation':'Statistical Analysis','Data mining':'Database Management'
                                              ,'data science':'Data Wrangling','Machine Learning':'Deep Learning','Business strategy':'Problem-Solving','MySQL':'Problem Solving',
                                              'Coding':'Big Data','orchestration':'Mathematics','C++':'Data Visualization','Computer science':'Data Wrangling'})

In [18]:
df['Skills List'].value_counts().head(10)

Skills List
Data analysis            1942
Data Analyst             1888
Consulting               1414
data visualization       1413
Data Wrangling           1413
Business intelligence    1259
big data                 1256
Problem-Solving          1099
Statistics               1099
Data                     1004
Name: count, dtype: int64

In [19]:
df['Location List'].value_counts().head(30)

Location List
Bengaluru                       12068
Chennai                          7011
Pune                             3858
Hyderabad                        3278
Gurugram                         2733
New Delhi                        2512
Noida                            2164
Mumbai (All Areas)               1664
Mumbai                           1378
Hybrid - Hyderabad               1264
Hybrid - Mumbai                  1264
Hosur                            1264
Kolkata                          1256
Delhi / NCR                       408
Hybrid - Kolkata                  400
Kochi                             350
Bengaluru(Bannerghatta Road)      212
Sakinaka                          100
India                              52
Karnataka                          50
Coimbatore                          9
Jaipur(MI Road)                     8
Surat                               8
Ahmedabad                           8
Hybrid - Pune                       8
Chandigarh                          

In [22]:
df.head(20)

,Link,Job Title,Company,Salary,Location,Experience,Skills,Day Posted,Skills List,Location List
0,https://www.naukri.com/job-listings-data-analy...,Data Analyst,AU Small Finance Bank,4-6 Lacs PA,Bengaluru,0-2 Yrs,"Analytics, R Studio, Data Extraction, Advanced...",2022-01,Analytics,Bengaluru
0,https://www.naukri.com/job-listings-data-analy...,Data Analyst,AU Small Finance Bank,4-6 Lacs PA,Bengaluru,0-2 Yrs,"Analytics, R Studio, Data Extraction, Advanced...",2022-01,R Studio,Bengaluru
0,https://www.naukri.com/job-listings-data-analy...,Data Analyst,AU Small Finance Bank,4-6 Lacs PA,Bengaluru,0-2 Yrs,"Analytics, R Studio, Data Extraction, Advanced...",2022-01,Data Extraction,Bengaluru
0,https://www.naukri.com/job-listings-data-analy...,Data Analyst,AU Small Finance Bank,4-6 Lacs PA,Bengaluru,0-2 Yrs,"Analytics, R Studio, Data Extraction, Advanced...",2022-01,Advanced Excel,Bengaluru
0,https://www.naukri.com/job-listings-data-analy...,Data Analyst,AU Small Finance Bank,4-6 Lacs PA,Bengaluru,0-2 Yrs,"Analytics, R Studio, Data Extraction, Advanced...",2022-01,Risk Modeling,Bengaluru
0,https://www.naukri.com/job-listings-data-analy...,Data Analyst,AU Small Finance Bank,4-6 Lacs PA,Bengaluru,0-2 Yrs,"Analytics, R Studio, Data Extraction, Advanced...",2022-01,Python,Bengaluru
0,https://www.naukri.com/job-listings-data-analy...,Data Analyst,AU Small Finance Bank,4-6 Lacs PA,Bengaluru,0-2 Yrs,"Analytics, R Studio, Data Extraction, Advanced...",2022-01,Microsoft Office Suite,Bengaluru
0,https://www.naukri.com/job-listings-data-analy...,Data Analyst,AU Small Finance Bank,4-6 Lacs PA,Bengaluru,0-2 Yrs,"Analytics, R Studio, Data Extraction, Advanced...",2022-01,Rstudio,Bengaluru
1,https://www.naukri.com/job-listings-client-dat...,Data Analyst,JPMorgan Chase Bank,2-6 Lacs PA,Bengaluru,0-5 Yrs,"Service delivery, Senior management, Product s...",2022-01,Service delivery,Bengaluru
1,https://www.naukri.com/job-listings-client-dat...,Data Analyst,JPMorgan Chase Bank,2-6 Lacs PA,Bengaluru,0-5 Yrs,"Service delivery, Senior management, Product s...",2022-01,Senior management,Bengaluru


In [23]:
df = df.drop('Link', axis=1)


In [24]:
df= df.drop('Skills',axis=1)

In [25]:
df= df.drop('Location',axis=1)
df.head()

,Job Title,Company,Salary,Experience,Day Posted,Skills List,Location List
0,Data Analyst,AU Small Finance Bank,4-6 Lacs PA,0-2 Yrs,2022-01,Analytics,Bengaluru
0,Data Analyst,AU Small Finance Bank,4-6 Lacs PA,0-2 Yrs,2022-01,R Studio,Bengaluru
0,Data Analyst,AU Small Finance Bank,4-6 Lacs PA,0-2 Yrs,2022-01,Data Extraction,Bengaluru
0,Data Analyst,AU Small Finance Bank,4-6 Lacs PA,0-2 Yrs,2022-01,Advanced Excel,Bengaluru
0,Data Analyst,AU Small Finance Bank,4-6 Lacs PA,0-2 Yrs,2022-01,Risk Modeling,Bengaluru


In [26]:
# df['Salary']= df['Salary'].replace({'50,000':'50000'})
def process_experience(exp_range):
    
    exp_range = str(exp_range)
    if '-' in exp_range:  # Handling range "2-7 Yrs"
        years = exp_range.split('-')
        lower = int(float(years[0].strip()))  # Convert to float first, then int
        upper = int(float(years[1].split()[0].strip()))  # Extract the upper bound
        return (lower + upper) / 2  # Return the average experience

    elif '+' in exp_range:  # Handling "3+ Yrs"
        return int(float(exp_range.split('+')[0].strip()))  # Convert to int safely
    
    else:  # Handling single value "5 Yrs"
        return int(float(exp_range.split()[0].strip()))  # Convert to int safely

def process_salary(salary_range):
    if pd.isna(salary_range):  # Handle NaN values
        return None  # Or set a default value like 0
    
    salary_range = str(salary_range).replace(',', '')  # Remove commas
    
    if '-' in salary_range:  # Handling range "30-35 Lacs PA"
        salary_parts = salary_range.split('-')
        lower = float(salary_parts[0].strip())  # Convert to float
        upper = float(salary_parts[1].split()[0].strip())  # Remove "Lacs PA"
        return (lower + upper) / 2  # Return the average salary
    
    else:  # Handling single value like "50 Lacs PA"
        return float(salary_range.split()[0].strip())  # Convert directly to float

# Apply function
df['Salary'] = df['Salary'].astype(str).apply(process_salary)
# Apply the functions
df['Experience'] = df['Experience'].apply(process_experience)
# df['Salary'] = df['Salary'].apply(process_salary)

# Check the processed DataFrame
print(df)

         Job Title                Company  Salary  Experience Day Posted  \
0     Data Analyst  AU Small Finance Bank     5.0         1.0    2022-01   
0     Data Analyst  AU Small Finance Bank     5.0         1.0    2022-01   
0     Data Analyst  AU Small Finance Bank     5.0         1.0    2022-01   
0     Data Analyst  AU Small Finance Bank     5.0         1.0    2022-01   
0     Data Analyst  AU Small Finance Bank     5.0         1.0    2022-01   
...            ...                    ...     ...         ...        ...   
4190  Data Analyst                 Zensar     4.0         3.0    2024-12   
4190  Data Analyst                 Zensar     4.0         3.0    2024-12   
4190  Data Analyst                 Zensar     4.0         3.0    2024-12   
4190  Data Analyst                 Zensar     4.0         3.0    2024-12   
4190  Data Analyst                 Zensar     4.0         3.0    2024-12   

            Skills List Location List  
0             Analytics     Bengaluru  
0      

In [27]:
df.head()

,Job Title,Company,Salary,Experience,Day Posted,Skills List,Location List
0,Data Analyst,AU Small Finance Bank,5.0,1.0,2022-01,Analytics,Bengaluru
0,Data Analyst,AU Small Finance Bank,5.0,1.0,2022-01,R Studio,Bengaluru
0,Data Analyst,AU Small Finance Bank,5.0,1.0,2022-01,Data Extraction,Bengaluru
0,Data Analyst,AU Small Finance Bank,5.0,1.0,2022-01,Advanced Excel,Bengaluru
0,Data Analyst,AU Small Finance Bank,5.0,1.0,2022-01,Risk Modeling,Bengaluru


In [28]:
df.to_csv('clean_ds.csv',index=False)

In [29]:
df['Experience'].value_counts()

Experience
3.5     8928
2.0     8800
0.5     4315
3.0     4124
4.0     3288
5.0     2686
4.5     2543
5.5     1833
7.0     1722
6.0     1656
2.5     1327
7.5     1172
6.5      458
1.5      456
1.0       17
15.0      16
8.0        9
8.5        8
9.5        8
Name: count, dtype: int64

In [30]:
df.shape

(43366, 7)